# 05_ngram_language_models: N-gram Models on Wikipedia Text
    
This notebook builds an N-gram Language Model from scratch and computes Perplexity metrics using a scraped Wikipedia text corpus.


In [1]:
import math
import requests
from bs4 import BeautifulSoup
import re
from collections import Counter, defaultdict

# 1. Scrape Wikipedia NLP Article
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(resp.content, "html.parser")
paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 80]
corpus_text = " ".join(paragraphs[:8]) # Take first 8 paragraphs

# Preprocess
corpus = re.sub(r"[^\w\s]", "", corpus_text).lower().split()
vocab = list(set(corpus))
vocab_size = len(vocab)
print(f"Corpus Token Count: {len(corpus)}, Vocabulary Size: {vocab_size}")

# 2. Count Unigrams and Bigrams
unigrams = Counter(corpus)
bigrams = Counter(zip(corpus[:-1], corpus[1:]))

# 3. Probability estimation with Laplace smoothing
def get_bigram_prob(w1, w2):
    count_bigram = bigrams[(w1, w2)]
    count_unigram = unigrams[w1]
    return (count_bigram + 1) / (count_unigram + vocab_size)

print("\nSmoothed transition probabilities:")
print("P(language | natural) =", get_bigram_prob("natural", "language"))
print("P(processing | natural) =", get_bigram_prob("natural", "processing"))

# 4. Calculate Perplexity on a test sequence
test_sequence = ["natural", "language", "processing", "methods", "and", "tasks"]

def compute_perplexity(seq):
    log_prob_sum = 0.0
    for i in range(1, len(seq)):
        w1, w2 = seq[i-1], seq[i]
        prob = get_bigram_prob(w1, w2)
        log_prob_sum += math.log(prob)
        
    avg_log_prob = log_prob_sum / (len(seq) - 1)
    return math.exp(-avg_log_prob)

ppl = compute_perplexity(test_sequence)
print(f"\nPerplexity of sequence {test_sequence}: {ppl:.4f}")


Corpus Token Count: 350, Vocabulary Size: 188

Smoothed transition probabilities:
P(language | natural) = 0.050761421319796954
P(processing | natural) = 0.005076142131979695

Perplexity of sequence ['natural', 'language', 'processing', 'methods', 'and', 'tasks']: 86.1401


### Output Explanation
- The model computes conditional sequence probabilities.
- Laplace smoothing prevents zero probabilities for unseen bigrams (like `"natural methods"`), keeping perplexity metrics finite and stable.
